<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day_19_EY_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile .env


Writing .env


In [2]:
!pip install -q langchain langchain-core langchain-community langchain-openai langchain-text-splitters faiss-cpu azure-search-documents azure-identity python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
import os

for name in [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
    "AZURE_OPENAI_EMBED_DEPLOYMENT",
    "AZURE_OPENAI_API_VERSION",
]:
    print(f"{name}: {os.getenv(name)}")

AZURE_OPENAI_ENDPOINT: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1
AZURE_OPENAI_CHAT_DEPLOYMENT: gpt-4o-1
AZURE_OPENAI_EMBED_DEPLOYMENT: text-embedding-3-small
AZURE_OPENAI_API_VERSION: 2024-10-21


In [8]:
import os
from openai import OpenAI

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ.get("AZURE_OPENAI_API_KEY")

print("Endpoint:", endpoint)
print("Embedding deployment:", os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"])

client = OpenAI(
    api_key=api_key,
    base_url=endpoint + "/"
)

response = client.embeddings.create(
    model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    input="hello"
)

print("Embedding dimension:", len(response.data[0].embedding))

Endpoint: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1
Embedding deployment: text-embedding-3-small
Embedding dimension: 1536


In [9]:
import os
from openai import AzureOpenAI
from langchain_openai import AzureOpenAIEmbeddings

def get_embeddings():
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
    api_key = os.environ["AZURE_OPENAI_API_KEY"]
    deployment = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]
    api_version = os.environ["AZURE_OPENAI_API_VERSION"]

    return AzureOpenAIEmbeddings(
        azure_endpoint=endpoint.split("/openai/")[0],
        api_key=api_key,
        azure_deployment=deployment,
        openai_api_version=api_version,
    )

In [10]:
emb = get_embeddings()

vector = emb.embed_query("hello")

print("Embedding dimension:", len(vector))
print("Azure OpenAI embeddings working!")

Embedding dimension: 1536
Azure OpenAI embeddings working!


In [11]:
emb = get_embeddings()
print("Embedding dimension:", len(emb.embed_query("hello")))

Embedding dimension: 1536


In [14]:
import os

print("Chat deployment:", os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"))
print("API version:", os.getenv("AZURE_OPENAI_API_VERSION"))

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
print("Endpoint:", endpoint)

Chat deployment: gpt-4o-1
API version: 2024-10-21
Endpoint: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1


In [15]:
import os
from openai import OpenAI

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ["AZURE_OPENAI_API_KEY"]
deployment = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]

client = OpenAI(
    api_key=api_key,
    base_url=endpoint + "/"
)

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "user", "content": "Reply with the single word: OK"}
    ],
    max_tokens=10
)

print("Chat deployment response:", response.choices[0].message.content)

Chat deployment response: OK


In [18]:
# Day 19 - Lab 1: Create the PolicyPilot document corpus

documents = [
    {
        "id": "HR-001",
        "title": "Work From Home Policy",
        "content": """
        Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.
        """
    },
    {
        "id": "TRV-002",
        "title": "Business Travel Policy",
        "content": """
        Business travel must be approved by the reporting manager
        before booking. Economy class should be used for domestic
        flights. Employees must submit travel expenses with receipts
        within seven working days after completing the trip.
        """
    },
    {
        "id": "SEC-003",
        "title": "Information Security Policy",
        "content": """
        Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.
        """
    },
    {
        "id": "CDP-004",
        "title": "Learning and Development Policy",
        "content": """
        Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.
        """
    },
    {
        "id": "PRC-005",
        "title": "Procurement Policy",
        "content": """
        Purchases must follow the company's approval matrix.
        Employees must obtain the required approvals before
        committing company funds. Vendor selection must follow
        applicable procurement and compliance requirements.
        """
    }
]

print("Number of policy documents:", len(documents))

for doc in documents:
    print(doc["id"], "-", doc["title"])

Number of policy documents: 5
HR-001 - Work From Home Policy
TRV-002 - Business Travel Policy
SEC-003 - Information Security Policy
CDP-004 - Learning and Development Policy
PRC-005 - Procurement Policy


In [19]:
%pip install -q langchain-text-splitters

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []

for doc in documents:
    doc_chunks = text_splitter.split_text(doc["content"])

    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "id": doc["id"],
            "title": doc["title"],
            "chunk_number": i + 1,
            "text": chunk
        })

print("Total chunks created:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

Total chunks created: 5

First chunk:
{'id': 'HR-001', 'title': 'Work From Home Policy', 'chunk_number': 1, 'text': 'Employees may work from home up to two days per week with\n        manager approval. Employees must remain available during\n        core business hours from 10 AM to 4 PM. Confidential company\n        information must only be accessed through approved systems.'}


In [21]:
%pip install -q faiss-cpu langchain-community

In [22]:
from langchain_core.documents import Document

department_map = {
    "HR-001": "HR",
    "TRV-002": "Finance",
    "SEC-003": "IT",
    "CDP-004": "Legal",
    "PRC-005": "Finance"
}

docs = []

for chunk in chunks:
    docs.append(
        Document(
            page_content=chunk["text"],
            metadata={
                "source": chunk["id"],
                "title": chunk["title"],
                "department": department_map[chunk["id"]],
                "chunk_id": f'{chunk["id"]}-{chunk["chunk_number"] - 1}'
            }
        )
    )

print("Documents prepared:", len(docs))
print("Sample metadata:", docs[0].metadata)

Documents prepared: 5
Sample metadata: {'source': 'HR-001', 'title': 'Work From Home Policy', 'department': 'HR', 'chunk_id': 'HR-001-0'}


In [25]:
from importlib.metadata import version

print("langchain-openai:", version("langchain-openai"))
print("openai:", version("openai"))

langchain-openai: 1.7.3
openai: 2.54.0


In [26]:
import os
from langchain_openai import OpenAIEmbeddings

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")

emb = OpenAIEmbeddings(
    model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    base_url=endpoint + "/",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

vector = emb.embed_query("Test Azure embeddings")

print("Embedding dimension:", len(vector))
print("Azure v1 embeddings working!")

Embedding dimension: 1536
Azure v1 embeddings working!


In [27]:
from langchain_community.vectorstores import FAISS

# Use the working Azure v1 embedding client
faiss_store = FAISS.from_documents(docs, emb)

print("FAISS vector database created successfully!")
print("Total policy chunks indexed:", len(docs))

FAISS vector database created successfully!
Total policy chunks indexed: 5


In [28]:
query = "What approval is required before booking business travel?"

results = faiss_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Title:", doc.metadata.get("title"))
    print("Content:", doc.page_content)


Result 1
Source: TRV-002
Title: Business Travel Policy
Content: Business travel must be approved by the reporting manager
        before booking. Economy class should be used for domestic
        flights. Employees must submit travel expenses with receipts
        within seven working days after completing the trip.

Result 2
Source: PRC-005
Title: Procurement Policy
Content: Purchases must follow the company's approval matrix.
        Employees must obtain the required approvals before
        committing company funds. Vendor selection must follow
        applicable procurement and compliance requirements.

Result 3
Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.


In [29]:
query = "What should employees do if they receive a phishing email?"

results = faiss_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Title:", doc.metadata.get("title"))
    print("Content:", doc.page_content)


Result 1
Source: SEC-003
Title: Information Security Policy
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

Result 2
Source: HR-001
Title: Work From Home Policy
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Result 3
Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.


In [30]:
%pip install -q azure-search-documents

In [32]:
from google.colab import userdata
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
)

endpoint = userdata.get("AZURE_SEARCH_ENDPOINT")
key = userdata.get("AZURE_SEARCH_KEY")

index_name = "policypilot-index"

fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True
    ),
    SearchableField(
        name="content",
        type=SearchFieldDataType.String
    ),
    SearchableField(
        name="source",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    SearchableField(
        name="title",
        type=SearchFieldDataType.String
    ),
    SearchableField(
        name="department",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(
            SearchFieldDataType.Single
        ),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="policy-vector-profile"
    ),
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(name="policy-hnsw")
    ],
    profiles=[
        VectorSearchProfile(
            name="policy-vector-profile",
            algorithm_configuration_name="policy-hnsw"
        )
    ]
)

index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search
)

index_client = SearchIndexClient(
    endpoint=endpoint,
    credential=__import__(
        "azure.core.credentials", fromlist=["AzureKeyCredential"]
    ).AzureKeyCredential(key)
)

result = index_client.create_or_update_index(index)

print("Index created successfully:", result.name)

Index created successfully: policypilot-index


In [34]:
existing_index = index_client.get_index("policypilot-index")

print("Existing index:", existing_index.name)
print("\nExisting fields:")

for field in existing_index.fields:
    print(
        f"Name: {field.name}, "
        f"Type: {field.type}, "
        f"Key: {field.key}, "
        f"Searchable: {field.searchable}, "
        f"Filterable: {field.filterable}"
    )

Existing index: policypilot-index

Existing fields:
Name: id, Type: SearchFieldDataType.STRING, Key: True, Searchable: False, Filterable: False
Name: content, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: False
Name: source, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: True
Name: title, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: False
Name: department, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: True
Name: content_vector, Type: Collection(Edm.Single), Key: False, Searchable: True, Filterable: False
